# PART 1 — Dataset Preparation (CIFAR-10)

## 📦 Task 1: Load and Inspect CIFAR-10

### Your Responsibilities
- Load the CIFAR-10 dataset
- Inspect image shapes and class labels
- Visualize sample images per class

---

## 🔧 Task 2: Preprocessing & DataLoaders

### Requirements
- Apply appropriate transforms:
  - Normalization
  - Data augmentation (for training set)
- Split data into:
  - Training
  - Validation
  - Test
- Build PyTorch `DataLoader`s

📌 **Deliverable:**  
Explain *why* each preprocessing step is used.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, Subset
####### 1. Load and inspect data
# Load dataset with basic Tensor conversion for inspection
basic_transform = transforms.Compose([transforms.ToTensor()])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=basic_transform)
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

"""
In this step, we load the dataset from pytorch directly, which is not common on real world applications
"""

# Inspection
image, label = train_set[0]
print(f"Image Shape: {image.shape}")  # Should be [3, 32, 32]
print(f"Label: {classes[label]}")

######### 2. Preprocessing and dataloaders
# Training: Augmentation + Normalization
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Validation/Test: Just Normalization
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
"""
 In this step, we use data augmentation, which is crucial for model generalizabilty and robustness
"""

# Load full datasets
full_train_data = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

print("Checking full train")
image, label = full_train_data[0]
print(f"Image Shape: {image.shape}")  # Should be [3, 32, 32]
print(f"Label: {classes[label]}")

# Shuffle the datasets to ensure labels deliverability:
np.random.seed(42)
indices = np.random.permutation(len(full_train_data))
full_train_data = Subset(full_train_data, indices)
# Split Training and Validation (90/10 split)
train_indices = list(range(45000))
val_indices = list(range(45000, 50000))

train_loader = DataLoader(Subset(full_train_data, train_indices), batch_size=64, shuffle=True)
val_loader = DataLoader(Subset(full_train_data, val_indices), batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

Image Shape: torch.Size([3, 32, 32])
Label: frog
Checking full train
Image Shape: torch.Size([3, 32, 32])
Label: frog



# PART 2 — Training a CNN From Scratch

## 🧠 Task 3: Design a CNN Architecture (From Scratch)

### Constraints
- No pretrained weights
- Architecture must include:
  - Convolution blocks
  - Non-linearities
  - Pooling
  - Regularization (Dropout / BatchNorm)

📌 **Goal:**  
Iteratively improve the architecture until performance **plateaus**.


## 🧪 Task 4: Scratch Training Experiments

### Experiments to Run
- Vary depth and width
- Try different optimizers
- Adjust learning rates and schedulers

### Record
- Best validation accuracy
- Training stability
- Signs of underfitting / overfitting

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# --- TASK 3: CNN ARCHITECTURE ---
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class CifarCNN(nn.Module):
    def __init__(self, input_size = 32):
        super(CifarCNN, self).__init__()
        # Block1 : 32x32 -> 16x16
        self.block1 = nn.Sequential(
            ConvBlock(3, 64),
            ConvBlock(64,64),
            nn.MaxPool2d(2,2),
        )

        # Block2: 16x16 -> 8x8
        self.block2 = nn.Sequential(
            ConvBlock(64,128),
            ConvBlock(128,128),
            nn.MaxPool2d(2,2),
        )

        # Block3: 8x8 -> 4x4
        self.block3 = nn.Sequential(
            ConvBlock(128, 256),
            ConvBlock(256,256),
            nn.MaxPool2d(2,2),
        )

        # Fully Connected
        # the dimensions is reduced by -> 2x2x2 = 8
        final_spatial_dim = input_size // 8
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256 * final_spatial_dim * final_spatial_dim, 512),
            nn.ReLU(),
            nn.Linear(512,10),
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        # print(x.shape)
        x = self.head(x)
        return x

# --- TASK 4: TRAINING SETUP ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CifarCNN(input_size=32).to(device)

criterion = nn.CrossEntropyLoss()
# Experimenting with Adam + Weight Decay for regularization
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
# Scheduler to reduce LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

def train_one_epoch(epoch, loader):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in tqdm(loader, desc=f"Epoch {epoch+1}"):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100. * correct / total

def validate(loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return val_loss / len(loader), 100. * correct / total

# --- EXECUTION ---
epochs = 3
best_acc = 0.0

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(epoch, train_loader)
    val_loss, val_acc = validate(val_loader)

    # Step the scheduler based on validation loss
    scheduler.step(val_loss)

    print(f"Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

print(f"🏁 Training Complete. Best Validation Accuracy: {best_acc:.2f}%")

Epoch 1: 100%|██████████| 704/704 [01:02<00:00, 11.24it/s]


Train Loss: 2.156 | Train Acc: 18.58%
Val Loss: 2.146 | Val Acc: 18.66%


Epoch 2: 100%|██████████| 704/704 [01:05<00:00, 10.75it/s]


Train Loss: 2.124 | Train Acc: 20.00%
Val Loss: 2.108 | Val Acc: 20.42%


Epoch 3: 100%|██████████| 704/704 [01:01<00:00, 11.40it/s]


Train Loss: 2.113 | Train Acc: 20.29%
Val Loss: 2.127 | Val Acc: 19.44%
🏁 Training Complete. Best Validation Accuracy: 20.42%



# PART 3 — Transfer Learning with Pretrained Models

## 🔁 Task 5: Select a Pretrained Model

Choose **one non-ResNet architecture** from `torchvision`, such as:
- VGG
- DenseNet
- MobileNet
- EfficientNet

Explain why you chose it.

---

## ⚙️ Task 6: Apply Transfer Learning Strategies

Implement and evaluate the following **independently**:

### Strategy A — Head-Only Fine-Tuning
- Freeze entire backbone
- Train classifier only

### Strategy B — Partial Fine-Tuning
- Unfreeze last few convolutional blocks
- Train classifier + selected layers

### Strategy C — Full Fine-Tuning
- Unfreeze entire network
- Fine-tune end-to-end

📌 Track:
- Accuracy
- Convergence speed
- Training stability

---


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

# --- CONFIGURATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
EPOCHS = 3

# --- TRAINING FUNCTION ---
def train_model(model, optimizer, train_loader, val_loader, epochs):
    criterion = nn.CrossEntropyLoss()
    best_acc = 0.0

    for epoch in range(epochs):
        # Training Phase
        model.train()
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation Phase
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_acc = 100. * correct / total
        print(f"Validation Accuracy: {val_acc:.2f}%")
        if val_acc > best_acc:
            best_acc = val_acc

    return best_acc

# --- SETUP EXPERIMENT FUNCTION ---
def setup_experiment(strategy="A", num_classes=10):
    model = models.mobilenet_v2(weights='DEFAULT')
    model.classifier[1] = nn.Linear(model.last_channel, num_classes)
    model = model.to(device)

    if strategy == "A":
        # Strategy A: Head-Only
        for param in model.parameters():
            param.requires_grad = False
        for param in model.classifier.parameters():
            param.requires_grad = True
        optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

    elif strategy == "B":
        # Strategy B: Partial
        for param in model.features.parameters():
            param.requires_grad = False
        # Unfreeze the last block (18) + classifier
        for param in model.features[18].parameters():
            param.requires_grad = True
        for param in model.classifier.parameters():
            param.requires_grad = True
        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

    elif strategy == "C":
        # Strategy C: Full
        for param in model.parameters():
            param.requires_grad = True
        optimizer = optim.Adam(model.parameters(), lr=1e-5)

    return model, optimizer

# --- EXPERIMENT EXECUTION LOOP ---
results = {}

for s in ["A", "B", "C"]:
    print(f"\n🚀 Starting Strategy {s}")
    model, optimizer = setup_experiment(strategy=s)

    # Running the experiment
    best_val_acc = train_model(model, optimizer, train_loader, val_loader, epochs=EPOCHS)
    results[s] = best_val_acc

# --- FINAL COMPARISON ---
print("\n" + "="*30)
print("FINAL EXPERIMENT RESULTS")
print("="*30)
for s, acc in results.items():
    print(f"Strategy {s}: {acc:.2f}%")


🚀 Starting Strategy A
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 60.8MB/s]
Epoch 1/3: 100%|██████████| 704/704 [00:23<00:00, 30.49it/s]


Validation Accuracy: 37.86%


Epoch 2/3: 100%|██████████| 704/704 [00:23<00:00, 30.58it/s]


Validation Accuracy: 38.94%


...

Validation Accuracy: 24.02%


Epoch 2/3: 100%|██████████| 704/704 [00:31<00:00, 22.47it/s]


Validation Accuracy: 33.54%


Epoch 3/3: 100%|██████████| 704/704 [00:32<00:00, 21.93it/s]


Validation Accuracy: 40.16%

FINAL EXPERIMENT RESULTS
Strategy A: 40.00%
Strategy B: 42.46%
Strategy C: 40.16%


# PART 4 — Progressive Transfer Learning (Main Experiment)

## 🧪 Experimental Design

### Progressive Fine-Tuning Pipeline

| Phase | Trainable Parameters | Objective |
|------|---------------------|-----------|
| Phase 1 | Classification head only | Learn task-specific boundaries |
| Phase 2 | Head + later backbone layers | Adapt high-level features |
| Phase 3 | Full network | Global refinement |

---

## 🔬 Task 7: Progressive Training Experiment

### Procedure
1. Initialize with ImageNet weights
2. Train head-only
3. Gradually unfreeze layers
4. Retain learned weights across phases

📌 **Important:**  
Do NOT reinitialize the model between phases.

---

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm

# --- REUSABLE TRAINING FUNCTION ---
def train_phase(model, optimizer, loader, val_loader, phase_name, epochs=5):
    print(f"\n>>> Starting {phase_name}")
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(loader, desc=f"{phase_name} - Epoch {epoch+1}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, pred = outputs.max(1)
                total += labels.size(0)
                correct += pred.eq(labels).sum().item()

        print(f"Phase: {phase_name} | Val Acc: {100.*correct/total:.2f}%")

# --- INITIALIZATION ---
# Using MobileNetV2 as established in Part 3
model = models.mobilenet_v2(weights='DEFAULT')
model.classifier[1] = nn.Linear(model.last_channel, 10)
model = model.to(device)

# ==========================================
# PHASE 1: Classification Head Only
# ==========================================
for param in model.features.parameters():
    param.requires_grad = False

optimizer_p1 = optim.Adam(model.classifier.parameters(), lr=1e-3)
train_phase(model, optimizer_p1, train_loader, val_loader, "PHASE 1 (Head Only)", epochs=5)

# ==========================================
# PHASE 2: Head + Later Backbone Layers
# ==========================================
# Unfreeze the last block (Block 18)
for param in model.features[18].parameters():
    param.requires_grad = True

# We use a lower learning rate now that we're touching the backbone
optimizer_p2 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
train_phase(model, optimizer_p2, train_loader, val_loader, "PHASE 2 (Partial Unfreeze)", epochs=5)

# ==========================================
# PHASE 3: Full Network (Global Refinement)
# ==========================================
for param in model.parameters():
    param.requires_grad = True

# Use the lowest learning rate for final refinement
optimizer_p3 = optim.Adam(model.parameters(), lr=1e-5)
train_phase(model, optimizer_p3, train_loader, val_loader, "PHASE 3 (Full Fine-Tune)", epochs=5)

print("\n✅ Progressive Training Complete!")


>>> Starting PHASE 1 (Head Only)


PHASE 1 (Head Only) - Epoch 1: 100%|██████████| 704/704 [00:23<00:00, 29.59it/s]


Phase: PHASE 1 (Head Only) | Val Acc: 39.28%


PHASE 1 (Head Only) - Epoch 2: 100%|██████████| 704/704 [00:22<00:00, 30.79it/s]


Phase: PHASE 1 (Head Only) | Val Acc: 38.96%


...

Phase: PHASE 3 (Full Fine-Tune) | Val Acc: 50.72%


PHASE 3 (Full Fine-Tune) - Epoch 4: 100%|██████████| 704/704 [00:34<00:00, 20.47it/s]


Phase: PHASE 3 (Full Fine-Tune) | Val Acc: 52.00%


PHASE 3 (Full Fine-Tune) - Epoch 5: 100%|██████████| 704/704 [00:34<00:00, 20.55it/s]


Phase: PHASE 3 (Full Fine-Tune) | Val Acc: 52.50%

✅ Progressive Training Complete!



## 📊 Task 8: Comparative Analysis

Compare:
- Best separate strategy
- Progressive fine-tuning result

Evaluate:
- Final accuracy
- Total epochs
- Learning curve smoothness
- Generalization gap

---

## 🧠 Interpretation

### Why Progressive Fine-Tuning Works

1. **Weight continuity** preserves task-relevant features  
2. **Gradual adaptation** minimizes representation shock  
3. **Efficient optimization** avoids redundant relearning  
4. **Improved feature–classifier alignment** enhances generalization  

---


## 📘 Task 9: Hands-On Machine Learning (Aurélien Géron)

### Chapter 12 — Neural Networks with PyTorch

#### Your Tasks
- Implement all **end-of-chapter exercises**
- Add:
  - Code
  - Explanations
  - Observations

📌 Integrate solutions directly into this notebook.


# PART 6 — Final Discussion

## 📌 When to Use Each Approach

| Scenario | Recommended Strategy | Reason |
|--------|---------------------|--------|
| Production systems | Progressive fine-tuning | Best accuracy & stability |
| Limited compute | Progressive fine-tuning | Efficient use of epochs |
| Academic ablations | Separate training | Clean isolation |
| Rapid baselines | Head-only | Fastest |

---

## 🎯 Final Conclusion

Progressive transfer learning provides a **realistic, efficient, and professional training paradigm** by **building on learned knowledge instead of discarding it**.

This notebook demonstrates that:

> **Gradual adaptation consistently outperforms isolated fine-tuning strategies**, especially in real-world scenarios.

---

## 🧠 Follow-up Research Questions

- Does progressive fine-tuning always outperform separate training?
- What is the optimal epoch allocation per phase?
- How does dataset size affect progressive fine-tuning benefits?
- How early should deeper layers be unfrozen?

---

## ✅ Deliverables Checklist

- [ ] CIFAR-10 pipeline (from scratch)
- [ ] Scratch CNN experiments
- [ ] Transfer learning (3 strategies)
- [ ] Progressive fine-tuning experiment
- [ ] Géron Chapter 12 exercises
- [ ] Comparative analysis & conclusions

---
